In [1]:
import sys

import vis_plotting as vis_plt
import EAVILS_processing
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as lines
import my_utils
import scipy
import os
import yaml
import copy
import pandas as pd
import bidict

import astropy.io.fits as fits
from astropy.coordinates import Angle
from astropy import units as u



# The following establishes a few initial parameters

In [15]:
#For now this is still necessary to handle pointing information. In future versions this should be contained as metadata in h5 files
metafits_folders = 'testing_tutorial/metafits/'

#Flagging the coarse band lines for the MWA. In general this parameter can be used for flagging coarse-band lines,
#or things like known FM or satellite contaminated channels
permanent_flags = my_utils.coarse_band_flagging()

#This shape_dict is taken from SSINS, with an additional "subTV" pseudo-channel added as a control "clean channel". 
#This subTV channel is quite simply the frequencies below which the digital TV channels live
shape_dict = my_utils.shape_dict('MWA',add_subTV=True)
#We only want TV channels, so we remove the 'center_packet_loss' frequency type
del shape_dict['center_packet_loss']
print(shape_dict)
#This quite simply sorts our frequency channels based on their minimum frequency
sorted_freq_ranges,sorted_freq_mins = EAVILS_processing.freq_range_sort(shape_dict)
#Establishes the correspondence between polarizations and polarization indices
pol_bidict = bidict.bidict({0:'XX',1:'YY',2:'XY',3:'YX'})


{'TV6': [174000000.0, 181000000.0], 'TV7': [181000000.0, 188000000.0], 'TV8': [188000000.0, 195000000.0], 'TV9': [195000000.0, 202000000.0], 'subTV': [167075000.0, 174000000.0]}


# The following shows how to process raw files into EAVILS h5 files. This compresses the data volume greatly by averaging along baselines.
We limit this to just one example observation in this case due to size constraints, but in general all desired data should be processed in this way as an initial step

In [16]:
all_obs_id_list = ['1160941144']
raw_data_folder = '<raw_data/in/this/folder>'

for obs_id in all_obs_id_list:
    vis_plt.vis_plotting(obs_id = obs_id,input_data_folder=raw_data_folder,output_path='testing_tutorial/')

[OUTPUT CHECKING]; found testing_tutorial/h5_files/1160941144_spectra_data_cross.h5, bypassing spectra outputs


In [6]:
input_directory = '/testing_tutorial/h5_files/'

## Here we establish a threshold based on some simple assumptions on what we want our 

In [7]:
scipy.stats.norm.isf(.001/(5*100)) #False positive rate/(num_freq_channels * approx number of meas. per pointing)

4.611382362302668

In [8]:
threshold = 4.61

In [9]:
EOR0_lists = ['10-21-2016_obsids-early_only',
'10-30-2016_obsids-early_only',
'11-21-2016_obsids',
'10-26-2016_obsids-early_only',
'11-12-2016_obsids',
'12-15-2016_obsids',
'10-19-2016_obsids-early_only',
'10-15-2016_obsids-early_only',
'11-17-2016_obsids',
'10-22-2016_obsids-early_only',
'10-17-2016_obsids-early_only',
'11-19-2016_obsids',
'10-28-2016_obsids-early_only']

EOR1_lists = ['10-19-2016_obsids-late_only',
'10-22-2016_obsids-late_only',
'10-30-2016_obsids-late_only',
'10-28-2016_obsids-late_only',
'10-15-2016_obsids-late_only',
'10-26-2016_obsids-late_only',
'10-21-2016_obsids-late_only',
'10-17-2016_obsids-late_only']

In [10]:
import importlib

In [11]:
importlib.reload(EAVILS_processing)

<module 'EAVILS_processing' from '/Users/elillesk/repos/SSINS/EAVILS/EAVILS_processing.py'>

In [12]:
#Sets the time and frequency dimensions of the coarse-graining. 
#When the number of times or frequencies doesn't divide evenly by 
time_dim = 8
freq_dim = 1

In [13]:
importlib.reload(EAVILS_processing)
all_lists =  EOR0_lists+EOR1_lists
all_lists = sorted(all_lists)
for list_title in all_lists:
    list_file = f'example_obs_lists_split/{list_title}.txt'
    print(list_file)
    EAVILS_processing.process_data(list_file=list_file,input_directory=input_directory,output_directory='testing_tutorial',time_dim=time_dim,freq_dim=freq_dim,pol_bidict=pol_bidict,shape_dict=shape_dict,permanent_flags=permanent_flags,pre_flag_on_SSINS_masks=True)

example_obs_lists_split/10-15-2016_obsids-early_only.txt
Output file testing_tutorial/10-15-2016_obsids-early_only/Tdim8_Fdim1_EAVILS_processed.h5 already exists
Exiting function. Change clobber to True to overwrite existing file.
example_obs_lists_split/10-15-2016_obsids-late_only.txt
Output file testing_tutorial/10-15-2016_obsids-late_only/Tdim8_Fdim1_EAVILS_processed.h5 already exists
Exiting function. Change clobber to True to overwrite existing file.
example_obs_lists_split/10-17-2016_obsids-early_only.txt
Output file testing_tutorial/10-17-2016_obsids-early_only/Tdim8_Fdim1_EAVILS_processed.h5 already exists
Exiting function. Change clobber to True to overwrite existing file.
example_obs_lists_split/10-17-2016_obsids-late_only.txt
Output file testing_tutorial/10-17-2016_obsids-late_only/Tdim8_Fdim1_EAVILS_processed.h5 already exists
Exiting function. Change clobber to True to overwrite existing file.
example_obs_lists_split/10-19-2016_obsids-early_only.txt
Output file testing_tut

In [14]:
processed_data_directory = 'testing_tutorial/'
list_titles = [file.split('.')[0] for file in os.listdir('example_obs_lists_split/') if len(file.split('.')[0] )>0]
list_titles = sorted(list_titles)
sky_field_list = ['EOR0','EOR1']
sky_field_list_association = {list_title:'EOR0' for list_title in EOR0_lists}
sky_field_list_association.update({list_title:'EOR1' for list_title in EOR1_lists})

In [ ]:
#Establishes the order in which polarizations should be subtracted
pol_subtraction_order = {(0,1),(1,0),(2,3),(3,2)}

In [ ]:
importlib.reload(EAVILS_processing)
array_dict, dof_ref_dict,sky_field_dict,source_list_dict = EAVILS_processing.create_data_arrays(processed_data_directory = processed_data_directory,list_titles=list_titles, time_dim=time_dim,freq_dim=freq_dim, shape_dict=shape_dict,sky_field_list_association=sky_field_list_association,pol_subtraction_order=pol_subtraction_order)

In [ ]:
importlib.reload(EAVILS_processing)
datafr = EAVILS_processing.create_data_frame(pol_bidict=pol_bidict,shape_dict=shape_dict,array_dict=array_dict,sky_field_dict=sky_field_dict,source_list_dict=source_list_dict,metafits_folders=metafits_folders)

In [ ]:
initial_estimated_pol_sub_stdv= {sky_field:{} for sky_field in sky_field_list}

In [ ]:
clean_channel = 'subTV'
clean_channel_dof = dof_ref_dict[clean_channel]
for sky_field in sky_field_list:
    clean_channel_pol_sub_series = datafr.query(f'{'sky_field'}==\'{sky_field}\' & {'freq_range'}==\'{clean_channel}\' & {'SSINS_flagged'}=={False}')['pol_sub']
    clean_channel_scaling = np.sqrt(np.var(clean_channel_pol_sub_series,ddof=1))

    for freq_range in sorted_freq_ranges:
        pol_sub_series = datafr.query(f'{'sky_field'}==\'{sky_field}\' & {'freq_range'}==\'{freq_range}\' & {'SSINS_flagged'}=={False}')['pol_sub']
        channel_dof = dof_ref_dict[freq_range]
        estimated_scaling =  clean_channel_scaling*np.sqrt(clean_channel_dof/channel_dof)
        initial_estimated_pol_sub_stdv[sky_field][freq_range] = estimated_scaling
    
        plt.yscale('log')
        
        plt.hist((pol_sub_series)/estimated_scaling,bins=np.arange(-10,10,.1),density=True,histtype='step')
        
        x = np.linspace(scipy.stats.norm.ppf(0.0001,scale=1),
                                                    scipy.stats.norm.ppf(0.9999,scale=1), 100)
        plt.plot(x, scipy.stats.norm.pdf(x,scale=1),
                'r-', lw=5, alpha=0.6, label='standard normal pdf')
        plt.title(f'{freq_range}, {sky_field}, scaling factor={np.round(estimated_scaling,3)}')
        plt.show()
        print(freq_range,estimated_scaling)

In [ ]:
initial_estimated_pol_sub_stdv

In [ ]:
iteration_count=0

datafr['pointing_flagged'] = False #This should already be true, but ensures it's the case
estimated_pol_sub_stdv = copy.deepcopy(initial_estimated_pol_sub_stdv)
while True:
    print('iterations:',iteration_count)
    prev_estimated_pol_sub_stdv = copy.deepcopy(estimated_pol_sub_stdv)
    
    for source_list in np.unique(datafr.source_list):
        print(source_list)
        sky_field = sky_field_list_association[source_list]

        for pointing in np.unique(datafr.pointing):
            for freq_range in sorted_freq_ranges:
                query_string = f'sky_field==\'{sky_field}\' & source_list==\'{source_list}\' & pointing=={pointing} & freq_range==\'{freq_range}\'& SSINS_flagged=={False}'
                sub_series = datafr.query(query_string)['pol_sub']
                estimated_stdv = estimated_pol_sub_stdv[sky_field][freq_range]
                outliers = np.abs(sub_series)>(estimated_stdv*threshold)
    
                if np.sum(outliers)>0:
                    flag_pointing=True
                else:
                    flag_pointing=False
                if flag_pointing:
                    flagged_indices = datafr.query(query_string).index
                    datafr.loc[flagged_indices,'pointing_flagged'] = True
    
    
    for sky_field in sky_field_list:
        for freq_range in sorted_freq_ranges:
            query_string_clean = f'sky_field==\'{sky_field}\' & freq_range==\'{freq_range}\'& SSINS_flagged=={False} & pointing_flagged=={False}'
            pol_sub_series_clean = datafr.query(query_string_clean)['pol_sub']
    
            scaling = np.sqrt(np.var(pol_sub_series_clean,ddof=1))
            
            estimated_pol_sub_stdv[sky_field][freq_range] = scaling

    print(estimated_pol_sub_stdv) 
    if estimated_pol_sub_stdv==prev_estimated_pol_sub_stdv:    
        for sky_field in sky_field_list:
            for freq_range in sorted_freq_ranges:
                query_string_all = f'sky_field==\'{sky_field}\' & freq_range==\'{freq_range}\'& SSINS_flagged=={False}'
                pol_sub_series_all = datafr.query(query_string_all)['pol_sub']
                
                query_string_clean = f'sky_field==\'{sky_field}\' & freq_range==\'{freq_range}\'& SSINS_flagged=={False} & pointing_flagged=={False}'
                pol_sub_series_clean = datafr.query(query_string_clean)['pol_sub']
                plt.yscale('log')
                
                scaling = estimated_pol_sub_stdv[sky_field][freq_range]
                
                plt.hist((pol_sub_series_clean)/scaling,bins=np.arange(-10,10,.1),density=True,histtype='step',label='all data (except SSINS flagged)')
                plt.hist((pol_sub_series_all)/scaling,bins=np.arange(-10,10,.1),density=True,histtype='step',label='clean data only')
                
                x = np.linspace(scipy.stats.norm.ppf(0.0001,scale=1),
                                                            scipy.stats.norm.ppf(0.9999,scale=1), 100)
                plt.plot(x, scipy.stats.norm.pdf(x,scale=1),
                        'r-', lw=5, alpha=0.6, label='standard normal pdf')
                plt.title(f'{freq_range}, {sky_field}, scaling factor={np.round(scaling,3)}')
                plt.show()
        break
    iteration_count+=1
           

In [ ]:
estimated_pol_sub_stdv

In [ ]:
datafr= EAVILS_processing.add_pol_sub_stdv(datafr,estimated_pol_sub_stdv)

In [ ]:
per_pointing_stats=EAVILS_processing.find_per_pointing_stats(datafr=datafr,shape_dict=shape_dict,list_titles=list_titles,threshold=threshold)


In [ ]:
estimated_pol_sub_stdv

In [ ]:
importlib.reload(EAVILS_processing)

integration_time=2 #2 second integration time
time_buffer =120 #approximate time between observations (just used for plotting spacing)
EAVILS_processing.create_plots(list_file =  'example_obs_lists_split/10-19-2016_obsids-late_only.txt',array_dict=array_dict,per_pointing_stats=per_pointing_stats,dof_ref_dict = dof_ref_dict,input_directory=input_directory,processed_data_directory=processed_data_directory, time_dim=time_dim,freq_dim=freq_dim,shape_dict=shape_dict,sky_field_list_association=sky_field_list_association,metafits_folders=metafits_folders,pol_bidict=pol_bidict,threshold=threshold,estimated_pol_sub_stdv = estimated_pol_sub_stdv,permanent_flags=permanent_flags,time_buffer=time_buffer,integration_time=integration_time)